In [ ]:
# ============================================================
# 1. IMPORT THƯ VIỆN
# ============================================================

import pandas as pd
from google.colab import files


# ============================================================
# 2. UPLOAD 3 FILE
#
# - inventory_new.csv
# - dim_product.csv
# - dim_date.csv
# ============================================================

uploaded = files.upload()


# ============================================================
# 3. TỰ ĐỘNG XÁC ĐỊNH FILE
# ============================================================

inventory_file = None
product_file = None
date_file = None


for filename in uploaded.keys():

    name = filename.lower()

    if "inventory_new" in name and name.endswith(".csv"):
        inventory_file = filename

    elif "dim_product" in name and name.endswith(".csv"):
        product_file = filename

    elif "dim_date" in name and name.endswith(".csv"):
        date_file = filename


if inventory_file is None:
    raise FileNotFoundError(
        "Không tìm thấy inventory_new.csv"
    )

if product_file is None:
    raise FileNotFoundError(
        "Không tìm thấy dim_product.csv"
    )

if date_file is None:
    raise FileNotFoundError(
        "Không tìm thấy dim_date.csv"
    )


# ============================================================
# 4. ĐỌC DỮ LIỆU
# ============================================================

inventory = pd.read_csv(
    inventory_file,
    low_memory=False
)

dim_product = pd.read_csv(
    product_file,
    low_memory=False
)

dim_date = pd.read_csv(
    date_file,
    low_memory=False
)


print(
    "INVENTORY:",
    inventory.shape
)

print(
    "DIM_PRODUCT:",
    dim_product.shape
)

print(
    "DIM_DATE:",
    dim_date.shape
)


# ============================================================
# 5. CHUẨN HÓA TÊN CỘT
# ============================================================

def normalize_columns(df):

    df = df.copy()

    df.columns = (
        df.columns
        .astype(str)
        .str.strip()
        .str.lower()
        .str.replace(
            r"[^\w]+",
            "_",
            regex=True
        )
        .str.strip("_")
    )

    return df


inventory = normalize_columns(
    inventory
)

dim_product = normalize_columns(
    dim_product
)

dim_date = normalize_columns(
    dim_date
)


# ============================================================
# 6. KIỂM TRA TÊN CỘT
# ============================================================

print(
    "\nINVENTORY columns:"
)

print(
    inventory.columns.tolist()
)

print(
    "\nDIM_PRODUCT columns:"
)

print(
    dim_product.columns.tolist()
)

print(
    "\nDIM_DATE columns:"
)

print(
    dim_date.columns.tolist()
)


# ============================================================
# 7. ĐỔI product_id_fk -> product_id
#
# inventory_new dùng product_id_fk
# DIM_PRODUCT dùng product_id
#
# Đổi tên để thuận tiện khi merge
# ============================================================

if (
    "product_id_fk" in inventory.columns
    and
    "product_id" not in inventory.columns
):

    inventory = inventory.rename(
        columns={
            "product_id_fk":
            "product_id"
        }
    )


# ============================================================
# 8. KIỂM TRA CỘT BẮT BUỘC
# ============================================================

required_inventory = [
    "snapshot_date",
    "product_id",
    "stock_on_hand",
    "units_received",
    "units_sold",
    "stockout_days",
    "days_of_supply",
    "fill_rate",
    "sell_through_rate",
    "stockout_flag",
    "overstock_flag",
    "reorder_flag"
]


required_product = [
    "product_key",
    "product_id"
]


required_date = [
    "date_key",
    "full_date"
]


for col in required_inventory:

    if col not in inventory.columns:

        raise KeyError(
            f"INVENTORY thiếu cột: {col}"
        )


for col in required_product:

    if col not in dim_product.columns:

        raise KeyError(
            f"DIM_PRODUCT thiếu cột: {col}"
        )


for col in required_date:

    if col not in dim_date.columns:

        raise KeyError(
            f"DIM_DATE thiếu cột: {col}"
        )


# ============================================================
# 9. CHUẨN HÓA NGÀY
# ============================================================

inventory["snapshot_date"] = pd.to_datetime(
    inventory["snapshot_date"],
    errors="coerce"
).dt.normalize()


dim_date["full_date"] = pd.to_datetime(
    dim_date["full_date"],
    errors="coerce"
).dt.normalize()


# ============================================================
# 10. CHUẨN HÓA product_id
#
# Đưa cả hai về String để merge tránh lỗi kiểu dữ liệu
# ============================================================

inventory["product_id"] = (
    inventory["product_id"]
    .astype("string")
    .str.strip()
)


dim_product["product_id"] = (
    dim_product["product_id"]
    .astype("string")
    .str.strip()
)


# ============================================================
# 11. CHUẨN HÓA MEASURE DẠNG SỐ
# ============================================================

numeric_columns = [
    "stock_on_hand",
    "units_received",
    "units_sold",
    "stockout_days",
    "days_of_supply",
    "fill_rate",
    "sell_through_rate"
]


for col in numeric_columns:

    inventory[col] = pd.to_numeric(
        inventory[col],
        errors="coerce"
    )


# ============================================================
# 12. CHUẨN HÓA BOOLEAN
# ============================================================

boolean_columns = [
    "stockout_flag",
    "overstock_flag",
    "reorder_flag"
]


def convert_boolean(value):

    if pd.isna(value):
        return pd.NA

    value = str(value).strip().lower()

    if value in [
        "true",
        "1",
        "yes",
        "y"
    ]:
        return True

    if value in [
        "false",
        "0",
        "no",
        "n"
    ]:
        return False

    return pd.NA


for col in boolean_columns:

    inventory[col] = (
        inventory[col]
        .apply(convert_boolean)
        .astype("boolean")
    )


# ============================================================
# 13. MAP snapshot_date -> snapshot_date_key
#
# inventory.snapshot_date
#          ↓
# DIM_DATE.full_date
#          ↓
# date_key
# ============================================================

inventory = inventory.merge(
    dim_date[
        [
            "date_key",
            "full_date"
        ]
    ],
    left_on="snapshot_date",
    right_on="full_date",
    how="left",
    validate="many_to_one"
)


inventory = inventory.rename(
    columns={
        "date_key":
        "snapshot_date_key"
    }
)


# ============================================================
# 14. MAP product_id -> product_key
#
# inventory.product_id
#          ↓
# DIM_PRODUCT.product_id
#          ↓
# product_key
# ============================================================

inventory = inventory.merge(
    dim_product[
        [
            "product_key",
            "product_id"
        ]
    ],
    on="product_id",
    how="left",
    validate="many_to_one"
)


# ============================================================
# 15. KIỂM TRA FOREIGN KEY
# ============================================================

missing_date_key = (
    inventory[
        "snapshot_date_key"
    ]
    .isna()
    .sum()
)


missing_product_key = (
    inventory[
        "product_key"
    ]
    .isna()
    .sum()
)


print(
    "\nSố snapshot_date_key không map được:",
    missing_date_key
)

print(
    "Số product_key không map được:",
    missing_product_key
)


# ============================================================
# 16. HIỂN THỊ CÁC DÒNG MAP KHÔNG ĐƯỢC
# ============================================================

if missing_date_key > 0:

    print(
        "\nCác snapshot_date không tồn tại trong DIM_DATE:"
    )

    display(
        inventory.loc[
            inventory[
                "snapshot_date_key"
            ].isna(),
            [
                "snapshot_date"
            ]
        ]
        .drop_duplicates()
        .head(20)
    )


if missing_product_key > 0:

    print(
        "\nCác product_id không tồn tại trong DIM_PRODUCT:"
    )

    display(
        inventory.loc[
            inventory[
                "product_key"
            ].isna(),
            [
                "product_id"
            ]
        ]
        .drop_duplicates()
        .head(20)
    )


# ============================================================
# 17. KIỂM TRA GRAIN
#
# Grain:
# Một dòng =
# Một sản phẩm tại một ngày snapshot
# ============================================================

duplicate_grain = (
    inventory
    .duplicated(
        subset=[
            "snapshot_date_key",
            "product_key"
        ]
    )
    .sum()
)


print(
    "\nSố dòng trùng "
    "snapshot_date_key + product_key:",
    duplicate_grain
)


# ============================================================
# 18. NẾU CÓ GRAIN BỊ TRÙNG
# HIỂN THỊ ĐỂ KIỂM TRA
#
# KHÔNG tự động xóa vì có thể làm mất dữ liệu.
# ============================================================

if duplicate_grain > 0:

    print(
        "\nCác dòng grain bị trùng:"
    )

    display(
        inventory[
            inventory
            .duplicated(
                subset=[
                    "snapshot_date_key",
                    "product_key"
                ],
                keep=False
            )
        ]
        .sort_values(
            [
                "snapshot_date_key",
                "product_key"
            ]
        )
        .head(30)
    )


# ============================================================
# 19. TẠO FACT_INVENTORY_SNAPSHOT
#
# Lấy đúng các cột trong StarModel.mmd
# ============================================================

fact_inventory_snapshot = inventory[
    [
        "snapshot_date_key",
        "product_key",
        "stock_on_hand",
        "units_received",
        "units_sold",
        "stockout_days",
        "days_of_supply",
        "fill_rate",
        "sell_through_rate",
        "stockout_flag",
        "overstock_flag",
        "reorder_flag"
    ]
].copy()


# ============================================================
# 20. TẠO inventory_snapshot_id
#
# Surrogate Primary Key
# ============================================================

fact_inventory_snapshot.insert(
    0,
    "inventory_snapshot_id",
    range(
        1,
        len(
            fact_inventory_snapshot
        ) + 1
    )
)


# ============================================================
# 21. CHUẨN HÓA KIỂU KHÓA
# ============================================================

fact_inventory_snapshot[
    "snapshot_date_key"
] = pd.to_numeric(
    fact_inventory_snapshot[
        "snapshot_date_key"
    ],
    errors="coerce"
).astype("Int64")


fact_inventory_snapshot[
    "product_key"
] = pd.to_numeric(
    fact_inventory_snapshot[
        "product_key"
    ],
    errors="coerce"
).astype("Int64")


# ============================================================
# 22. KIỂM TRA MISSING VALUE CUỐI CÙNG
# ============================================================

print(
    "\n===== MISSING VALUE ====="
)

print(
    fact_inventory_snapshot
    .isna()
    .sum()
)


# ============================================================
# 23. KIỂM TRA KẾT QUẢ
# ============================================================

print(
    "\n===== FACT_INVENTORY_SNAPSHOT ====="
)


print(
    "Số dòng:",
    len(
        fact_inventory_snapshot
    )
)


print(
    "Số product_key:",
    fact_inventory_snapshot[
        "product_key"
    ].nunique()
)


print(
    "Số snapshot_date_key:",
    fact_inventory_snapshot[
        "snapshot_date_key"
    ].nunique()
)


display(
    fact_inventory_snapshot.head(20)
)


# ============================================================
# 24. XUẤT FILE
# ============================================================

fact_inventory_snapshot.to_csv(
    "fact_inventory_snapshot.csv",
    index=False
)


print(
    "\nĐã tạo file fact_inventory_snapshot.csv"
)


